# 📊 Exploratory Data Analysis (EDA)
## Prediksi Tingkat Burnout Mahasiswa

Notebook ini bertujuan untuk melakukan analisis eksploratif (EDA) lengkap pada dataset `student_mental_health_burnout.csv`.

**Tahapan EDA:**
1. Load Dataset
2. Informasi Dataset (Shape, Info, Missing Values, Duplicates)
3. Statistik Deskriptif
4. Visualisasi Data (Countplot, Histogram, Boxplot, Heatmap, Pairplot)


In [ ]:
# Import semua library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

print("✅ Library berhasil diimport!")

---
## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('dataset/student_mental_health_burnout.csv')
print(f"Dataset berhasil dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom")
df.head(10)

---
## 2. Informasi Dataset

In [ ]:
# Shape dataset
print("=" * 40)
print(f"📐 Shape Dataset: {df.shape}")
print(f"   - Jumlah baris  : {df.shape[0]:,}")
print(f"   - Jumlah kolom  : {df.shape[1]}")
print("=" * 40)

In [ ]:
# Informasi tipe data setiap kolom
print("📋 Informasi Dataset:")
df.info()

In [ ]:
# Cek Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ Tidak ada missing values dalam dataset!")
else:
    print("⚠️ Kolom dengan missing values:")
    print(missing_df)

In [ ]:
# Cek Duplikat
dup_count = df.duplicated().sum()
print(f"🔁 Jumlah baris duplikat: {dup_count}")
if dup_count == 0:
    print("✅ Tidak ada data duplikat!")

---
## 3. Statistik Deskriptif

In [ ]:
# Statistik deskriptif untuk kolom numerik
print("📊 Statistik Deskriptif Kolom Numerik:")
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Distribusi kolom kategorik
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Kolom kategorik: {cat_cols}")
print()
for col in cat_cols:
    print(f"📌 {col}:")
    print(df[col].value_counts())
    print()

---
## 4. Visualisasi Data

### 4.1 Distribusi Target: Burnout Level

> **Insight**: Grafik ini menunjukkan keseimbangan distribusi kelas pada variabel target `burnout_level`. Distribusi yang seimbang penting agar model tidak bias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
order = ['Low', 'Medium', 'High']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
sns.countplot(data=df, x='burnout_level', order=order, palette=colors, ax=axes[0])
axes[0].set_title('Distribusi Burnout Level (Countplot)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Burnout Level')
axes[0].set_ylabel('Jumlah Mahasiswa')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=10, fontweight='bold')

# Pie Chart
counts = df['burnout_level'].value_counts()[order]
axes[1].pie(counts, labels=order, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporsi Burnout Level (Pie Chart)', fontsize=13, fontweight='bold')

plt.suptitle('Distribusi Variabel Target: Burnout Level', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/burnout_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Distribusi kelas Burnout Level cukup merata antara Low, Medium, dan High.")
print("Hal ini mengindikasikan dataset yang balanced sehingga model tidak akan bias.")

### 4.2 Histogram Fitur Numerik

> **Insight**: Histogram memperlihatkan distribusi setiap fitur numerik. Distribusi yang normal atau mendekati normal menandakan fitur yang lebih stabil untuk dijadikan predictor.

In [ ]:
# Kolom numerik (kecuali student_id)
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
if 'student_id' in numerical_cols:
    numerical_cols = numerical_cols.drop('student_id')

n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Frekuensi')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
    axes[i].legend(fontsize=8)

# Sembunyikan subplot yang tidak terpakai
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur Numerik (Histogram)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/histograms.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Garis merah menunjukkan nilai rata-rata (mean) setiap fitur.")
print("Fitur seperti daily_sleep_hours dan cgpa menunjukkan distribusi yang relatif normal.")

### 4.3 Boxplot - Deteksi Outlier

> **Insight**: Boxplot digunakan untuk mendeteksi pencilan (outlier). Titik-titik di luar 'whisker' (garis panjang) adalah data outlier.

In [ ]:
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df['burnout_level'], y=df[col],
                order=['Low', 'Medium', 'High'],
                palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'},
                ax=axes[i])
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Burnout Level')
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot Fitur Numerik per Kategori Burnout', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/boxplots.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Boxplot dikelompokkan berdasarkan Burnout Level untuk melihat perbedaan distribusi.")
print("Perbedaan yang signifikan antar grup menunjukkan fitur tersebut berpengaruh terhadap burnout.")

### 4.4 Correlation Heatmap

> **Insight**: Heatmap korelasi menunjukkan hubungan linear antar fitur numerik. Nilai mendekati +1 = korelasi positif kuat, mendekati -1 = korelasi negatif kuat.

In [ ]:
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Tampilkan segitiga bawah saja
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    center=0
)
plt.title('Correlation Heatmap Fitur Numerik', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('images/heatmap.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Heatmap hanya menampilkan segitiga bawah untuk menghindari redundansi.")
print("Perhatikan fitur mana yang berkorelasi tinggi dengan fitur lain (potensi multikolinearitas).")

### 4.5 Analisis Hubungan Antar Fitur dengan Burnout Level

> Pairplot digunakan untuk melihat hubungan antar beberapa fitur penting sekaligus, diwarnai berdasarkan kelas burnout.

In [ ]:
# Sampling 2000 baris untuk efisiensi
sample_df = df.sample(n=2000, random_state=42)

selected_features = ['cgpa', 'anxiety_score', 'depression_score', 'daily_sleep_hours', 'screen_time_hours', 'burnout_level']

g = sns.pairplot(
    sample_df[selected_features],
    hue='burnout_level',
    hue_order=['Low', 'Medium', 'High'],
    palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'},
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 20}
)
g.fig.suptitle('Pairplot Fitur Penting per Kategori Burnout', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('images/pairplot.png', bbox_inches='tight', dpi=100)
plt.show()

print("\n📝 Penjelasan:")
print("Titik warna merah (High) yang terpisah dari hijau (Low) mengindikasikan fitur diskriminatif yang baik.")
print("Fitur seperti depression_score dan anxiety_score tampak memisahkan kelas dengan cukup baik.")

---
## 5. Analisis Tambahan: Hubungan Fitur Kunci dengan Burnout

### 5.1 Jam Tidur vs Burnout

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot jam tidur per burnout level
sns.boxplot(data=df, x='burnout_level', y='daily_sleep_hours',
            order=['Low', 'Medium', 'High'],
            palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'},
            ax=axes[0])
axes[0].set_title('Jam Tidur per Burnout Level', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Burnout Level')
axes[0].set_ylabel('Jam Tidur per Hari')

# Rata-rata jam tidur
sleep_avg = df.groupby('burnout_level')['daily_sleep_hours'].mean()[['Low', 'Medium', 'High']]
axes[1].bar(sleep_avg.index, sleep_avg.values,
            color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black')
axes[1].set_title('Rata-rata Jam Tidur per Burnout Level', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Burnout Level')
axes[1].set_ylabel('Rata-rata Jam Tidur')
for bar, val in zip(axes[1].patches, sleep_avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Hubungan Jam Tidur dengan Burnout Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/sleep_vs_burnout.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Mahasiswa dengan burnout High cenderung memiliki jam tidur yang lebih rendah.")
print("Ini mengkonfirmasi bahwa kurang tidur berkorelasi positif dengan tingkat burnout yang lebih tinggi.")

### 5.2 Screen Time vs Burnout

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='burnout_level', y='screen_time_hours',
               order=['Low', 'Medium', 'High'],
               palette={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'},
               inner='box')
plt.title('Distribusi Screen Time per Burnout Level (Violin Plot)', fontsize=13, fontweight='bold')
plt.xlabel('Burnout Level')
plt.ylabel('Screen Time (Jam/Hari)')
plt.tight_layout()
plt.savefig('images/screentime_vs_burnout.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("Violin plot menunjukkan distribusi screen time di setiap kategori burnout.")
print("Mahasiswa dengan burnout High cenderung memiliki screen time yang lebih bervariasi.")

### 5.3 CGPA vs Burnout

In [ ]:
plt.figure(figsize=(10, 5))
for level, color in zip(['Low', 'Medium', 'High'], ['#2ecc71', '#f39c12', '#e74c3c']):
    subset = df[df['burnout_level'] == level]['cgpa']
    plt.hist(subset, bins=30, alpha=0.6, label=f'{level} ({len(subset):,})', color=color, edgecolor='white')

plt.title('Distribusi CGPA per Burnout Level', fontsize=13, fontweight='bold')
plt.xlabel('CGPA')
plt.ylabel('Frekuensi')
plt.legend(title='Burnout Level')
plt.tight_layout()
plt.savefig('images/cgpa_vs_burnout.png', bbox_inches='tight', dpi=120)
plt.show()

print("\n📝 Penjelasan:")
print("CGPA tampaknya terdistribusi merata di semua tingkat burnout, mengindikasikan")
print("bahwa IPK tinggi saja tidak menjamin mahasiswa bebas dari burnout.")

---
## ✅ Ringkasan Temuan EDA

| Temuan | Detail |
|---|---|
| **Dataset** | 150.000 data mahasiswa, 20 kolom, tidak ada missing values |
| **Target** | Distribusi burnout_level cukup seimbang |
| **Jam Tidur** | Mahasiswa burnout High cenderung tidur lebih sedikit |
| **CGPA** | CGPA terdistribusi merata di semua kelas burnout |
| **Outlier** | Beberapa fitur seperti screen_time_hours memiliki outlier |
| **Korelasi** | anxiety_score dan depression_score berkorelasi positif |
